# H15b: Explicit SVD Clamping vs. Muon — Does Artificially Fixing Conditioning Match Muon's Loss?

## Scientific Context

**Parent experiment:** H15 measured condition numbers (kappa = sigma_max / sigma_min) across optimizers and found a striking paradox: Muon sometimes achieves **worse** conditioning than SGD (kappa ratio ~0.8x) while simultaneously delivering **3x better loss**. This directly challenges the popular narrative that Muon works *because* it improves weight matrix conditioning.

## The Falsification Logic

If conditioning improvement is the mechanism by which Muon achieves lower loss, then **artificially forcing good conditioning on SGD** (via explicit SVD clamping) should recover Muon-level performance. Conversely, if forced-conditioning SGD still cannot match Muon, then the mechanism must lie elsewhere — specifically in the **directional quality** of the polar factor update, not in its side-effect on singular value spectra.

## What This Experiment Does

We compare four optimizer variants on identical networks:

| Method | Description |
|--------|-------------|
| **SGD** | Standard SGD with momentum — baseline |
| **Muon** | Newton-Schulz polar factor projection of gradients |
| **SGD + SVD Clamp** | After each SGD step, decompose W = U S V^T, clamp singular values so kappa <= target, recompose |
| **SGD + SVD Equalize** | After each SGD step, set ALL singular values to mean(S) — perfect kappa = 1 |

If neither clamping nor equalization can match Muon, the conditioning hypothesis is **falsified**.

## Key Predictions

- **If conditioning is the mechanism:** SVD-clamped SGD (especially at low kappa targets like 2 or 5) should approach Muon's loss. SVD-equalized (kappa=1, the theoretical optimum) should match or beat Muon.
- **If direction quality is the mechanism:** Neither intervention should close the gap significantly, because they fix the spectrum but leave the gradient *direction* unchanged — SGD still walks in the raw gradient direction, just with post-hoc spectral surgery.

In [ ]:
"""
H15b: Explicit SVD Clamping -- Does Improving Conditioning Match Muon's Loss?
==============================================================================

MOTIVATION (from H15 surprise):
  Matrix layers under Muon sometimes have WORSE kappa than SGD (0.8x) despite
  3x better loss. If conditioning improvement is NOT the mechanism, then
  artificially forcing good conditioning should NOT match Muon's loss.

QUESTION: If we add explicit SVD clamping to SGD (clamp sigma_max/sigma_min
  to target kappa after each step), does it match Muon's loss trajectory?
  If NO: confirms direction quality, not conditioning, is the mechanism.
  If YES: conditioning IS the mechanism and H15 results were confounded.

PROTOCOL:
  Optimizers:
    (a) SGD -- baseline
    (b) Muon -- polar factor
    (c) SGD + SVD clamping -- after each SGD step, decompose W=USV^T,
        clamp S so kappa(W) <= target_kappa, recompose W.
    (d) SGD + SVD equalize -- set ALL SVs to mean(S) after each step.
  Sweep target_kappa for (c) in {2, 5, 10, 50}.

KEY TESTS:
  T1: Does SVD-clamped SGD (kappa<=5) match Muon's final loss within 2x?
  T2: Does SVD-equalized SGD (all SVs equal) match Muon?
  T3: If both fail, the conditioning-path explanation is falsified.

Setup: 4-layer, 32x32, 500 steps, 10 seeds, LR swept per method.
"""
print("H15b: Explicit SVD Clamping vs. Muon")
print("=" * 60)
print()
print("This experiment is a FALSIFICATION TEST for the hypothesis")
print("that Muon's advantage comes from improving weight matrix")
print("conditioning (kappa = sigma_max / sigma_min).")
print()
print("If artificially forcing kappa -> 1 via SVD surgery on SGD")
print("does NOT match Muon's loss, then conditioning is NOT the")
print("mechanism. The polar factor's directional quality is.")

## Section 1: Imports and Dependencies

Standard numerical computing only. No deep learning frameworks needed -- this is a controlled linear-network experiment where we can compute exact gradients analytically.

In [ ]:
import numpy as np
import os

print(f"NumPy version: {np.__version__}")
print("All computation is pure NumPy -- exact gradients, no autograd needed.")

## Section 2: Experimental Configuration

The design space is carefully chosen:
- **DIM = 32**: Large enough for meaningful singular value spectra (32 SVs per layer), small enough for exact SVD every step.
- **NUM_LAYERS = 4**: Deep enough that conditioning effects compound through the product W4 W3 W2 W1, but shallow enough to remain tractable.
- **NUM_STEPS = 500**: Sufficient for convergence differences to manifest clearly.
- **KAPPA_TARGETS = [2, 5, 10, 50]**: Spans from aggressive clamping (kappa=2, nearly orthogonal) to mild (kappa=50, barely constrained). If ANY target matches Muon, the conditioning hypothesis survives.

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
print(f"Script directory: {SCRIPT_DIR}")

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 10
BATCH_SIZE = 64

print("--- Experimental Hyperparameters ---")
print(f"  Weight matrix dimension:      {DIM} x {DIM}  ({DIM**2} parameters per layer)")
print(f"  Network depth:                {NUM_LAYERS} layers  ({NUM_LAYERS * DIM**2} total parameters)")
print(f"  Training steps:               {NUM_STEPS}")
print(f"  Momentum coefficient:         {MOMENTUM}")
print(f"  Newton-Schulz iterations:     {NS_ITERS}  (for Muon's polar factor approximation)")
print(f"  Number of seeds:              {NUM_SEEDS}  (for statistical reliability)")
print(f"  Batch size:                   {BATCH_SIZE}  (fixed input/output pairs per seed)")
print()
print("NOTE: Each seed generates a DIFFERENT random regression problem (X, Y)")
print("      AND different initial weights. Results are averaged across seeds.")

In [ ]:
LR_CANDIDATES = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.002, 0.001]
KAPPA_TARGETS = [2, 5, 10, 50]

print("--- Learning Rate Sweep Grid ---")
print(f"  LR candidates: {LR_CANDIDATES}")
print(f"  Total LRs to try per method: {len(LR_CANDIDATES)}")
print()
print("--- SVD Clamping Kappa Targets ---")
for kt in KAPPA_TARGETS:
    print(f"  kappa <= {kt:>3}  =>  sigma_min >= sigma_max / {kt}")
    if kt == 2:
        print(f"               (aggressive: SVs within 2x of each other)")
    elif kt == 5:
        print(f"               (moderate: the key test target)")
    elif kt == 50:
        print(f"               (mild: barely constraining)")
print()
print("Additionally: SVD-equalize forces ALL SVs to mean(S), giving kappa = 1 exactly.")

## Section 3: Core Algorithms — Newton-Schulz, SVD Clamping, SVD Equalization

Three key transformations are used in this experiment:

### Newton-Schulz Iteration (Muon's polar factor approximation)
Given gradient G, compute its polar factor U_p such that G = U_p H (polar decomposition). The Newton-Schulz iteration converges to U_p via: X_{k+1} = (3/2) X_k - (1/2) X_k (X_k^T X_k). This replaces the gradient direction with one that treats all singular value directions equally -- it "democratizes" the update across the spectrum.

### SVD Clamping
After each SGD step, decompose W = U diag(s) V^T. For each singular value s_i, enforce s_i >= s_max / kappa_target. This directly controls conditioning without changing the left/right singular vectors (the "directions" in weight space). The gradient direction itself is untouched.

### SVD Equalization
The extreme version of clamping: set ALL singular values to mean(s). This achieves kappa = 1 (perfect conditioning) while preserving the singular vector structure. If conditioning were the whole story, this should be optimal.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

# Demonstrate Newton-Schulz on a random matrix
demo_M = np.random.RandomState(0).randn(DIM, DIM)
demo_polar = newton_schulz(demo_M)
demo_svs = np.linalg.svd(demo_polar, compute_uv=False)
print("Newton-Schulz demonstration on random 32x32 matrix:")
print(f"  Input Frobenius norm:   {np.linalg.norm(demo_M, ord='fro'):.4f}")
print(f"  Output singular values: min={demo_svs.min():.6f}, max={demo_svs.max():.6f}")
print(f"  Output kappa:           {demo_svs.max()/demo_svs.min():.4f}")
print(f"  (Ideal polar factor has all SVs = 1.0, kappa = 1.0)")
print(f"  Convergence quality:    {'GOOD' if demo_svs.max()/demo_svs.min() < 1.1 else 'NEEDS MORE ITERS'}")
del demo_M, demo_polar, demo_svs

In [ ]:
def svd_clamp(W, target_kappa):
    """Clamp singular values so kappa(W) <= target_kappa."""
    U, s, Vt = np.linalg.svd(W, full_matrices=False)
    s_max = s[0]
    s_min_target = s_max / target_kappa
    s_clamped = np.maximum(s, s_min_target)
    return U @ np.diag(s_clamped) @ Vt

# Demonstrate SVD clamping
demo_W = np.random.RandomState(1).randn(DIM, DIM)
demo_svs_before = np.linalg.svd(demo_W, compute_uv=False)
print("SVD Clamping demonstration:")
print(f"  Before: kappa = {demo_svs_before[0]/demo_svs_before[-1]:.2f}  "
      f"(sigma_max={demo_svs_before[0]:.3f}, sigma_min={demo_svs_before[-1]:.3f})")
for kt in KAPPA_TARGETS:
    demo_clamped = svd_clamp(demo_W, kt)
    demo_svs_after = np.linalg.svd(demo_clamped, compute_uv=False)
    n_lifted = np.sum(demo_svs_before < demo_svs_before[0] / kt)
    print(f"  After kappa<={kt:>2}: kappa = {demo_svs_after[0]/demo_svs_after[-1]:.2f}  "
          f"(sigma_min lifted from {demo_svs_before[-1]:.3f} to {demo_svs_after[-1]:.3f}, "
          f"{n_lifted} SVs modified)")
del demo_W, demo_svs_before, demo_clamped, demo_svs_after

In [ ]:
def svd_equalize(W):
    """Set all SVs to mean(SVs)."""
    U, s, Vt = np.linalg.svd(W, full_matrices=False)
    s_eq = np.full_like(s, np.mean(s))
    return U @ np.diag(s_eq) @ Vt

# Demonstrate SVD equalization
demo_W = np.random.RandomState(2).randn(DIM, DIM)
demo_svs_before = np.linalg.svd(demo_W, compute_uv=False)
demo_eq = svd_equalize(demo_W)
demo_svs_after = np.linalg.svd(demo_eq, compute_uv=False)
print("SVD Equalization demonstration:")
print(f"  Before: kappa = {demo_svs_before[0]/demo_svs_before[-1]:.2f}")
print(f"          SV range: [{demo_svs_before.min():.3f}, {demo_svs_before.max():.3f}]")
print(f"  After:  kappa = {demo_svs_after[0]/demo_svs_after[-1]:.6f}")
print(f"          ALL SVs = {demo_svs_after[0]:.6f} (= mean of original SVs)")
print(f"  Key insight: equalization preserves LEFT and RIGHT singular vectors")
print(f"               (the directional structure) but forces perfect conditioning.")
print(f"               If conditioning is the whole story, this is the best possible fix.")
del demo_W, demo_svs_before, demo_eq, demo_svs_after

## Section 4: Network Architecture — Deep Linear Network

The network computes y = W_4 W_3 W_2 W_1 x (a deep linear network). Despite being "linear" in the mathematical sense (the composition is still a linear map), deep linear networks exhibit highly nonlinear optimization landscapes due to the multiplicative interaction of weight matrices. This makes them an ideal testbed because:

1. **Exact gradients** are available analytically -- no stochastic minibatch noise.
2. **Conditioning effects compound**: kappa(W_4 W_3 W_2 W_1) can grow as the product of individual kappas.
3. **Singular value dynamics** are well-studied theoretically, letting us compare against known results.

Weights are initialized as I + 0.1 * noise, starting near the identity with mild perturbation.

In [ ]:
def init_weights(seed):
    rng = np.random.RandomState(seed)
    return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(NUM_LAYERS)]

# Show initial conditioning of the network
demo_weights = init_weights(42)
print("Initial weight matrix properties (seed=42):")
for i, W in enumerate(demo_weights):
    svs = np.linalg.svd(W, compute_uv=False)
    print(f"  Layer {i}: kappa={svs[0]/svs[-1]:.3f}, "
          f"SV range=[{svs.min():.3f}, {svs.max():.3f}], "
          f"Frobenius={np.linalg.norm(W, 'fro'):.3f}")

# Show the product conditioning (what the network "sees")
prod = demo_weights[0]
for W in demo_weights[1:]:
    prod = W @ prod
prod_svs = np.linalg.svd(prod, compute_uv=False)
print(f"  Product W4*W3*W2*W1: kappa={prod_svs[0]/prod_svs[-1]:.3f}")
print(f"  NOTE: Product kappa can grow multiplicatively with depth.")
del demo_weights, prod, prod_svs

## Section 5: Forward Pass, Loss, and Gradient Computation

The forward pass is a simple chain of matrix multiplications: y = W_L ... W_2 W_1 x. The loss is mean squared error: L = (1/2N) sum_i ||y_i - t_i||^2. Gradients are computed via backpropagation through the linear chain, which reduces to exact matrix calculus:

grad_l = delta @ activations[l]^T, where delta propagates backward via delta <- W[l]^T @ delta.

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

## Section 6: Training Loop — The Heart of the Comparison

The training loop implements all four optimizer variants in a single function, selected by the `method` parameter. The critical structural difference:

- **SGD and Muon** differ in how they TRANSFORM the gradient before the update step. Muon applies Newton-Schulz to extract the polar factor, changing the gradient *direction*. SGD uses the raw gradient.
- **SVD Clamp and SVD Equalize** use the SAME gradient direction as SGD (raw gradient with momentum), but perform post-hoc surgery on the weight matrix's singular value spectrum AFTER each step.

This is the key experimental design: Muon changes the *step direction* while clamping/equalization changes the *weight spectrum*. If Muon's advantage is about spectrum, clamping should replicate it. If it is about direction, clamping cannot replicate it.

In [ ]:
def train(weights_init, X, Y, lr, method, kappa_target=None):
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for i in range(NUM_LAYERS):
            if method == 'muon':
                mom[i] = MOMENTUM * mom[i] + newton_schulz(grads[i])
                weights[i] = weights[i] - lr * mom[i]
            elif method == 'sgd':
                mom[i] = MOMENTUM * mom[i] + grads[i]
                weights[i] = weights[i] - lr * mom[i]
            elif method == 'sgd_clamp':
                mom[i] = MOMENTUM * mom[i] + grads[i]
                weights[i] = weights[i] - lr * mom[i]
                weights[i] = svd_clamp(weights[i], kappa_target)
            elif method == 'sgd_equalize':
                mom[i] = MOMENTUM * mom[i] + grads[i]
                weights[i] = weights[i] - lr * mom[i]
                weights[i] = svd_equalize(weights[i])
    return compute_loss(weights, X, Y)

print("Training loop defined. Key method dispatch:")
print("  'sgd'          ->  W -= lr * (momentum * prev + grad)")
print("  'muon'         ->  W -= lr * (momentum * prev + newton_schulz(grad))")
print("  'sgd_clamp'    ->  SGD step, THEN clamp SVs so kappa <= target")
print("  'sgd_equalize' ->  SGD step, THEN set ALL SVs to mean(SVs)")
print()
print("NOTE: Clamping/equalization happens EVERY STEP -- 500 SVD decompositions")
print("      per layer per run. This is computationally expensive but gives the")
print("      conditioning hypothesis its best possible chance.")

## Section 7: Data Generation and Learning Rate Sweep

Each seed generates a fixed random regression problem: X (inputs) and Y (targets), both drawn from N(0, 0.3^2). The LR sweep uses 3 seeds for efficiency, selecting the LR that minimizes mean final loss across those seeds. This ensures each method gets its best possible LR -- a fair comparison.

In [ ]:
def make_data(seed):
    rng = np.random.RandomState(seed)
    X = rng.randn(DIM, BATCH_SIZE) * 0.3
    Y = rng.randn(DIM, BATCH_SIZE) * 0.3
    return X, Y

In [ ]:
def sweep_lr(method, seeds, kappa_target=None):
    best_lr, best_loss = LR_CANDIDATES[-1], float('inf')
    for lr in LR_CANDIDATES:
        losses = []
        for s in seeds:
            X, Y = make_data(s)
            w = init_weights(s + 5000)
            fl = train(w, X, Y, lr, method, kappa_target)
            losses.append(fl)
        ml = np.mean([l for l in losses if np.isfinite(l)]) if any(np.isfinite(l) for l in losses) else float('inf')
        if ml < best_loss:
            best_loss = ml
            best_lr = lr
    return best_lr

## Section 8: Main Experiment — LR Sweep, Full Training, and Hypothesis Testing

The experiment proceeds in three phases:

1. **Phase 1 (LR Sweep):** For each optimizer variant, sweep 10 learning rates using 3 seeds, select the best LR. This is critical for fairness -- SVD-clamped methods may prefer different LRs than vanilla SGD.

2. **Phase 2 (Full Training):** Train all 10 seeds at the best LR for each method. Record final losses.

3. **Phase 3 (Hypothesis Tests):** Apply the three pre-registered tests:
   - **T1:** Does SVD-clamped SGD (kappa<=5) achieve loss within 2x of Muon?
   - **T2:** Does SVD-equalized SGD (kappa=1) achieve loss within 2x of Muon?
   - **T3:** If both T1 and T2 fail, the conditioning-path explanation is falsified.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H15b: EXPLICIT SVD CLAMPING -- Does Improving kappa Match Muon's Loss?")
    print("=" * 100)
    print(f"Network: {NUM_LAYERS}-layer, {DIM}x{DIM}, {NUM_STEPS} steps, {NUM_SEEDS} seeds")
    print(f"Kappa targets for clamped SGD: {KAPPA_TARGETS}")
    print(f"Seeds: {seeds}")
    print()

    # -------------------------------------------------------------------------
    # PHASE 1: LR SWEEPS
    # -------------------------------------------------------------------------
    print("=" * 100)
    print("PHASE 1: Learning Rate Sweeps")
    print("=" * 100)
    print()
    print("Each method gets its own optimal LR. This is critical for fairness:")
    print("SVD-clamped methods may need different LRs because the spectral surgery")
    print("interacts with the step size (clamping effectively adds a projection step")
    print("that could amplify or dampen the effect of large learning rates).")
    print()

    configs = [('sgd', None), ('muon', None), ('sgd_equalize', None)]
    for kt in KAPPA_TARGETS:
        configs.append(('sgd_clamp', kt))

    print(f"Total optimizer configurations: {len(configs)}")
    for method, kt in configs:
        name = method if kt is None else f"{method}_k{kt}"
        print(f"  - {name}")
    print()

    best_lrs = {}
    for method, kt in configs:
        name = method if kt is None else f"{method}_k{kt}"
        lr = sweep_lr(method, seeds[:3], kt)
        best_lrs[name] = lr
        print(f"  {name:>20}: best_lr={lr:.4f}")

    print()
    print("--- LR Sweep Summary ---")
    print("Observation: Compare the best LRs across methods.")
    print("If clamped methods need very different LRs, it suggests the spectral")
    print("surgery is significantly changing the optimization dynamics.")
    sgd_lr = best_lrs['sgd']
    muon_lr = best_lrs['muon']
    print(f"  SGD best LR:  {sgd_lr:.4f}")
    print(f"  Muon best LR: {muon_lr:.4f}  (ratio to SGD: {muon_lr/sgd_lr:.2f}x)")
    for kt in KAPPA_TARGETS:
        clamp_lr = best_lrs[f'sgd_clamp_k{kt}']
        print(f"  Clamp k={kt:>2} LR: {clamp_lr:.4f}  (ratio to SGD: {clamp_lr/sgd_lr:.2f}x)")
    eq_lr = best_lrs['sgd_equalize']
    print(f"  Equalize LR:  {eq_lr:.4f}  (ratio to SGD: {eq_lr/sgd_lr:.2f}x)")

    # -------------------------------------------------------------------------
    # PHASE 2: FULL TRAINING
    # -------------------------------------------------------------------------
    print()
    print("=" * 100)
    print("PHASE 2: Full Training (all 10 seeds)")
    print("=" * 100)
    print()
    print("Training each configuration with its best LR across all seeds...")
    print()

    results = {}
    results_per_seed = {}
    for method, kt in configs:
        name = method if kt is None else f"{method}_k{kt}"
        lr = best_lrs[name]
        losses = []
        for s in seeds:
            X, Y = make_data(s)
            w = init_weights(s + 5000)
            fl = train(w, X, Y, lr, method, kt)
            losses.append(fl)
        finite = [l for l in losses if np.isfinite(l)]
        results[name] = np.mean(finite) if finite else float('inf')
        results_per_seed[name] = losses

        # Print per-method summary
        n_finite = len(finite)
        n_diverged = len(losses) - n_finite
        print(f"  {name:>20}  lr={lr:.4f}  "
              f"mean_loss={results[name]:.6e}  "
              f"std={np.std(finite):.6e}  "
              f"min={min(finite):.6e}  max={max(finite):.6e}  "
              f"({n_finite}/{len(losses)} converged"
              f"{', '+str(n_diverged)+' DIVERGED' if n_diverged > 0 else ''})")

    # -------------------------------------------------------------------------
    # PHASE 3: RESULTS AND ANALYSIS
    # -------------------------------------------------------------------------
    print()
    print(f"{'=' * 100}")
    print("RESULTS")
    print(f"{'=' * 100}")

    muon_loss = results['muon']
    sgd_loss = results['sgd']

    print()
    print("--- Raw Loss Comparison ---")
    print(f"  {'Method':>20}  {'Loss':>14}  {'vs Muon':>10}  {'vs SGD':>10}")
    print("  " + "-" * 58)
    for name in ['sgd', 'muon', 'sgd_equalize'] + [f'sgd_clamp_k{kt}' for kt in KAPPA_TARGETS]:
        r = results[name]
        vs_muon = r / max(muon_loss, 1e-30)
        vs_sgd = r / max(sgd_loss, 1e-30)
        print(f"  {name:>20}  {r:>14.6e}  {vs_muon:>10.2f}x  {vs_sgd:>10.2f}x")

    print()
    print("--- Interpretation of Loss Table ---")
    print(f"  Muon achieves {muon_loss:.6e} loss.")
    print(f"  SGD achieves  {sgd_loss:.6e} loss.")
    print(f"  Gap: Muon is {sgd_loss/max(muon_loss,1e-30):.1f}x better than SGD.")
    print()

    # Analyze the clamping sweep
    print("  SVD Clamping trend (does tighter kappa help?):")
    for kt in KAPPA_TARGETS:
        cl = results[f'sgd_clamp_k{kt}']
        gap_to_muon = cl / max(muon_loss, 1e-30)
        gap_to_sgd = cl / max(sgd_loss, 1e-30)
        direction = "BETTER than SGD" if cl < sgd_loss else "WORSE than SGD"
        print(f"    kappa<={kt:>2}: loss={cl:.6e} ({gap_to_muon:.2f}x Muon, {gap_to_sgd:.2f}x SGD) -- {direction}")

    eq_loss = results.get('sgd_equalize', float('inf'))
    eq_vs_muon = eq_loss / max(muon_loss, 1e-30)
    print(f"\n  SVD Equalize (kappa=1): loss={eq_loss:.6e} ({eq_vs_muon:.2f}x Muon)")
    if eq_loss > muon_loss * 2:
        print("  => Even PERFECT conditioning cannot match Muon!")
        print("     This is strong evidence that conditioning is NOT the mechanism.")
    else:
        print("  => Perfect conditioning approaches Muon -- conditioning MAY matter.")

    # -------------------------------------------------------------------------
    # HYPOTHESIS TESTS
    # -------------------------------------------------------------------------
    print(f"\n\n{'=' * 100}")
    print("HYPOTHESIS TESTS")
    print(f"{'=' * 100}")

    clamp5_loss = results.get('sgd_clamp_k5', float('inf'))

    t1 = clamp5_loss < muon_loss * 2
    print(f"\n  T1: SVD-clamped SGD (kappa<=5) within 2x of Muon?")
    print(f"      Clamped: {clamp5_loss:.6e}, Muon: {muon_loss:.6e}, ratio: {clamp5_loss/max(muon_loss,1e-30):.2f}x")
    print(f"      --> {'PASS (conditioning explains it)' if t1 else 'FAIL (conditioning insufficient)'}")
    if not t1:
        print(f"      Interpretation: Even forcing kappa<=5 leaves a {clamp5_loss/max(muon_loss,1e-30):.1f}x gap to Muon.")
        print(f"      The gradient DIRECTION matters more than the weight SPECTRUM.")

    t2 = eq_loss < muon_loss * 2
    print(f"\n  T2: SVD-equalized SGD within 2x of Muon?")
    print(f"      Equalized: {eq_loss:.6e}, Muon: {muon_loss:.6e}, ratio: {eq_loss/max(muon_loss,1e-30):.2f}x")
    print(f"      --> {'PASS' if t2 else 'FAIL'}")
    if not t2:
        print(f"      Interpretation: Even PERFECT conditioning (kappa=1.0 every step)")
        print(f"      leaves a {eq_loss/max(muon_loss,1e-30):.1f}x gap. This is the strongest possible")
        print(f"      evidence against the conditioning hypothesis.")

    t3 = not t1 and not t2
    print(f"\n  T3: If T1 and T2 both fail, conditioning-path explanation is falsified.")
    print(f"      --> {'CONFIRMED: direction quality, not conditioning' if t3 else 'NOT YET FALSIFIED'}")
    if t3:
        print(f"\n      CONCLUSION: Muon's advantage is NOT about improving kappa.")
        print(f"      The polar factor projection changes the gradient DIRECTION")
        print(f"      (equalizing how much each singular mode contributes to the update)")
        print(f"      in a way that cannot be replicated by post-hoc spectral surgery.")
        print(f"      This is consistent with the 'spectral democracy' / 'gauge-fixing'")
        print(f"      interpretation: Muon re-weights the gradient, not the weights.")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions and Scientific Interpretation

### What This Experiment Tells Us

H15b is a **controlled ablation** that isolates the two candidate mechanisms for Muon's advantage:

1. **Conditioning improvement** (kappa reduction): Muon might work because it keeps weight matrices well-conditioned, preventing gradient signals from being distorted by extreme singular value ratios.

2. **Directional quality** (polar factor): Muon might work because the Newton-Schulz polar factor replaces the raw gradient with one that treats all singular directions equally, regardless of their current scale.

### The Falsification Structure

The experiment is designed as a strong falsification test:

- If SVD clamping (which DIRECTLY controls kappa) matches Muon, then conditioning IS the mechanism.
- If SVD equalization (which achieves PERFECT kappa=1) matches Muon, then conditioning IS the mechanism.
- If NEITHER matches Muon despite giving the conditioning hypothesis every possible advantage (perfect kappa, optimal LR, same number of steps), then **conditioning is falsified as the primary mechanism**.

### Why This Matters for the Broader Theory

This connects to the **Muon-as-RG-Gauge-Fixing** framework:

- The "gauge-fixing" interpretation says Muon removes a *redundancy* in how gradients are expressed -- the raw gradient over-represents already-large singular directions and under-represents small ones. The polar factor eliminates this bias.
- SVD clamping, by contrast, fixes the *weights* but leaves the *gradient direction* biased. Even with perfect weight conditioning, the next gradient step will still over-update the dominant directions.
- This is why fixing the weights (clamping) is fundamentally different from fixing the gradient (polar projection): one is a **post-hoc correction** that gets undone each step, the other is a **proactive reweighting** of the optimization direction.

### Relationship to Other Experiments

- **H15** (parent): Found Muon has worse kappa than SGD -- motivated this follow-up.
- **H15a**: Measured "direction quality" directly, confirming Muon's gradient alignment is superior.
- **H3/H3a/H3b**: Partial SV equalization experiments that approach this from the other direction.
- **H6a/H6b**: LR confound audits ensuring the gap is not a learning rate artifact.